In [2]:
library(GenomicRanges)
library(data.table)
library(rtracklayer)

Warning message:
“package ‘GenomicRanges’ was built under R version 4.3.3”
Loading required package: S4Vectors

Warning message:
“package ‘S4Vectors’ was built under R version 4.3.3”

Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Warning message:
“package ‘IRanges’ was built under R version 4.3.3”
Loading required package: GenomeInfoDb

Warning message:
“package ‘GenomeInfoDb’ was built under R version 4.3.2”
Warning message:
“package ‘data.table’ was built under R version 4.3.3”

Attaching package: ‘data.table’


The following object is masked from ‘package:GenomicRanges’:

    shift


The following object is masked from ‘package:IRanges’:

    shift


The following objects are masked from ‘package:S4Vectors’:

    first, second


Warning message:
“package ‘rtracklayer’ was built under R version 4.3.3”


## Older

In [ ]:
# ================================================================
# DHS & ATAC-Seq Quality Control Pipeline (v2)
# ================================================================

# ------------------------------------------------
# Utility helpers
# ------------------------------------------------
safe_quantile <- function(x, q, na.rm = TRUE) {
  if (length(x) == 0 || all(is.na(x))) return(NA_real_)
  quantile(x, q, na.rm = na.rm)
}

width_check <- function(start, end, minw, maxw) {
  w <- end - start
  w >= minw & w <= maxw
}

summary_stats <- function(df, label) {
  cat(sprintf(
    "\n[%s] mean_signal: %.3f (IQR %.3f–%.3f), score: %.3f (IQR %.3f–%.3f)\n",
    label,
    mean(df$mean_signal, na.rm = TRUE),
    quantile(df$mean_signal, 0.25, na.rm = TRUE),
    quantile(df$mean_signal, 0.75, na.rm = TRUE),
    mean(df$score, na.rm = TRUE),
    quantile(df$score, 0.25, na.rm = TRUE),
    quantile(df$score, 0.75, na.rm = TRUE)
  ))
}

# ------------------------------------------------
# 1. DHS QC
# ------------------------------------------------
dhs_path <- "../ref/CRE_sites/DHS_sites/Cardiac.bed"
dhs_df   <- fread(dhs_path, sep="\t", header=TRUE)
cat("DHS - Initial:", nrow(dhs_df), "peaks\n")

# sanity: remove rows missing coords
dhs_df <- dhs_df[!is.na(Start) & !is.na(End) & Start < End]
dhs_df <- unique(dhs_df)
summary_stats(dhs_df, "DHS (raw)")

# Filter 1: Signal strength (top 80%)
signal_thr <- safe_quantile(dhs_df$mean_signal, 0.2)
dhs_df <- dhs_df[mean_signal > signal_thr]
cat("After signal filter:", nrow(dhs_df), "\n")

# Filter 2: Score threshold (>0.1)
dhs_df <- dhs_df[score > 0.1]
cat("After score filter:", nrow(dhs_df), "\n")

# Filter 3: Width filter (50–1000 bp)
dhs_df <- dhs_df[width_check(Start, End, 50, 1000)]
cat("After width filter:", nrow(dhs_df), "\n")

# Optional: remove blacklisted regions if you have them
blk_path <- "../ref/hg19-blacklist.v2.bed"
if (file.exists(blk_path)) {
  blk <- import(blk_path)
  gr_dhs <- GRanges(seqnames=dhs_df$Chrom,
                    ranges=IRanges(start=dhs_df$Start, end=dhs_df$End))
  gr_dhs <- subsetByOverlaps(gr_dhs, blk, invert=TRUE)
  dhs_df <- dhs_df[seq_len(length(gr_dhs)), ]
  cat("After blacklist removal:", nrow(dhs_df), "\n")
}

summary_stats(dhs_df, "DHS (filtered)")

#fwrite(dhs_df, "../ref/DHS_sites/Cardiac_filtered.bed", sep="\t")

# ------------------------------------------------
# 2. ATAC-Seq QC
# ------------------------------------------------
atac_path <- "../ref/CRE_sites/scATAC_sites/Adrenal_Cortical.bed"
atac_df   <- fread(atac_path, sep="\t", header=TRUE)
cat("\nATAC - Initial:", nrow(atac_df), "peaks\n")

atac_df <- atac_df[!is.na(Start) & !is.na(End) & Start < End]
atac_df <- unique(atac_df)
summary_stats(atac_df, "ATAC (raw)")

# Filter 1: Fold-change > 3
atac_df <- atac_df[fold.change > 3]
cat("After fold-change filter:", nrow(atac_df), "\n")

# Filter 2: FDR significance (log10qvalue > 1.3, i.e. q<0.05)
atac_df <- atac_df[log10qvalue > 1.3]
cat("After q-value filter:", nrow(atac_df), "\n")

# Filter 3: Score threshold (top 75%)
score_thr <- safe_quantile(atac_df$score, 0.25)
atac_df <- atac_df[score > score_thr]
cat("After score filter:", nrow(atac_df), "\n")

# Filter 4: Width 100–1000 bp
atac_df <- atac_df[width_check(Start, End, 100, 1000)]
cat("After width filter:", nrow(atac_df), "\n")

# Optional: adult-tissue presence
if ("Present.in.adult.tissues" %in% names(atac_df))
  atac_df <- atac_df[Present.in.adult.tissues == "yes"]

summary_stats(atac_df, "ATAC (filtered)")

#fwrite(atac_df, "../ref/scATAC_sites/Adrenal_Cortical_filtered.bed", sep="\t")

# ------------------------------------------------
# 3. QC Summary
# ------------------------------------------------
cat("\n=== QC SUMMARY ===\n")
cat(sprintf("DHS peaks retained: %d / %d (%.1f%%)\n",
            nrow(dhs_df), nrow(fread(dhs_path)),
            100*nrow(dhs_df)/nrow(fread(dhs_path))))
cat(sprintf("ATAC peaks retained: %d / %d (%.1f%%)\n",
            nrow(atac_df), nrow(fread(atac_path)),
            100*nrow(atac_df)/nrow(fread(atac_path))))

# ------------------------------------------------
# 4. Convert to GRanges and save .rds for pipeline
# ------------------------------------------------
to_gr <- function(df, name_label) {
  gr <- GRanges(
    seqnames = df$Chrom,
    ranges   = IRanges(start=df$Start, end=df$End),
    strand   = "*"
  )
  mcols(gr) <- DataFrame(name=name_label, score=df$score)
  gr
}

gr_dhs  <- to_gr(dhs_df,  "Cardiac_DHS")
gr_atac <- to_gr(atac_df, "Adrenal_ATAC")

#saveRDS(gr_dhs,  "../ref/DHS_sites/Cardiac_filtered.hg19.rds")
#saveRDS(gr_atac, "../ref/scATAC_sites/Adrenal_Cortical_filtered.hg19.rds")

cat("\n✅ DHS and ATAC GRanges objects saved for pipeline integration.\n")


In [1]:
#!/usr/bin/env Rscript
# ================================================================
# DHS & ATAC-Seq Quality Control Pipeline (v7 – Adaptive + Metadata)
# ================================================================
suppressPackageStartupMessages({
  library(data.table)
  library(GenomicRanges)
  library(rtracklayer)
})

# ------------------------------------------------
# Utility helpers
# ------------------------------------------------
safe_quantile <- function(x, q, na.rm = TRUE) {
  if (length(x) == 0 || all(is.na(x))) return(NA_real_)
  quantile(x, q, na.rm = na.rm)
}

width_check <- function(start, end, minw, maxw) {
  w <- end - start
  w >= minw & w <= maxw
}

width_stats <- function(start, end) {
  w <- end - start
  list(median_width = median(w), iqr_width = IQR(w))
}

summary_stats <- function(df, label) {
  has_ms <- "mean_signal" %in% names(df) && is.numeric(df$mean_signal)
  ms_mean <- if (has_ms) mean(df$mean_signal, na.rm = TRUE) else NA_real_
  ms_q25  <- if (has_ms) quantile(df$mean_signal, 0.25, na.rm = TRUE) else NA_real_
  ms_q75  <- if (has_ms) quantile(df$mean_signal, 0.75, na.rm = TRUE) else NA_real_
  sc_mean <- mean(df$score, na.rm = TRUE)
  sc_q25  <- quantile(df$score, 0.25, na.rm = TRUE)
  sc_q75  <- quantile(df$score, 0.75, na.rm = TRUE)
  sprintf("[%s] mean_signal: %.3f (IQR %.3f–%.3f), score: %.3f (IQR %.3f–%.3f)",
          label, ms_mean, ms_q25, ms_q75, sc_mean, sc_q25, sc_q75)
}

# ---- preserve metadata ------------------------------------------
to_gr <- function(df, name_label) {
  gr <- GRanges(seqnames = df$Chrom,
                ranges   = IRanges(start=df$Start, end=df$End),
                strand   = "*")
  meta_cols <- setdiff(names(df), c("Chrom", "Start", "End"))
  mcols(gr) <- as.data.frame(df[, meta_cols, with = FALSE])
  mcols(gr)$CRE_source <- name_label
  gr
}

# ------------------------------------------------
# Input directories
# ------------------------------------------------
dhs_dir  <- "../ref/CRE_sites/DHS_sites2"
atac_dir <- "../ref/CRE_sites/scATAC_sites2"
dhs_files  <- list.files(dhs_dir,  pattern="\\.bed$", full.names=TRUE)
atac_files <- list.files(atac_dir, pattern="\\.bed$", full.names=TRUE)

# ------------------------------------------------
# Output directories
# ------------------------------------------------
qc_root <- "../ref/CRE_sites_QC"
qc_dhs  <- file.path(qc_root, "DHS_sites_QC2")
qc_atac <- file.path(qc_root, "scATAC_sites_QC2")
dir.create(qc_root, recursive = TRUE, showWarnings = FALSE)
dir.create(qc_dhs,  recursive = TRUE, showWarnings = FALSE)
dir.create(qc_atac, recursive = TRUE, showWarnings = FALSE)

# ------------------------------------------------
# Load blacklist (optional)
# ------------------------------------------------
blk_path <- "../ref/hg19-blacklist.v2.bed"
blk <- if (file.exists(blk_path)) import(blk_path) else NULL
if (!is.null(blk)) cat("✅ Blacklist loaded:", length(blk), "regions\n")

# ------------------------------------------------
# Initialize outputs
# ------------------------------------------------
qc_summary <- data.table(
  dataset = character(),
  tissue = character(),
  initial_peaks = integer(),
  retained_peaks = integer(),
  retained_pct = numeric(),
  used_quantile = numeric(),
  median_width = numeric(),
  iqr_width = numeric(),
  mean_signal_filtered = numeric(),
  mean_score_filtered = numeric()
)

report_path <- file.path(qc_root, "QC_report2.txt")
sink(report_path, append = FALSE)
cat("=========== DHS & ATAC QC REPORT ===========\n\n")

# ------------------------------------------------
# Adaptive QC function
# ------------------------------------------------
process_cre <- function(path, type) {
  df <- fread(path, sep="\t", header=TRUE)
  base_name <- tools::file_path_sans_ext(basename(path))
  cat(sprintf("\n--- %s QC: %s ---\n", type, base_name))

  n0 <- nrow(df)
  df <- df[!is.na(Start) & !is.na(End) & Start < End]
  df <- unique(df)
  cat(summary_stats(df, paste(type, "(raw)")), "\n")

  # ------------------------------------------------
  # INITIAL FILTERS
  # ------------------------------------------------
  if (type == "DHS") {
    q_sig <- 0.10
    sig_thr <- safe_quantile(df$mean_signal, q_sig)

    q_score_dhs <- 0.10
    score_thr   <- safe_quantile(df$score, q_score_dhs)

    cat(sprintf("DHS thresholds (initial): mean_signal > %.4f ; score > %.4f\n",
                sig_thr, score_thr))

    n_a <- nrow(df)
    df1 <- df[mean_signal > sig_thr]
    cat(sprintf("  After mean_signal: %d (%.1f%%)\n", nrow(df1), 100*nrow(df1)/n_a))
    df2 <- df1[score > score_thr]
    cat(sprintf("  After score:       %d (%.1f%%)\n", nrow(df2), 100*nrow(df2)/n_a))
    df3 <- df2[width_check(Start, End, 40, 1500)]
    cat(sprintf("  After width:       %d (%.1f%%)\n", nrow(df3), 100*nrow(df3)/n_a))
    df <- df3

  } else if (type == "ATAC") {
    q_score <- 0.25
    df <- df[fold.change > 3 & log10qvalue > 1.3]
    score_thr <- safe_quantile(df$score, q_score)
    df <- df[score > score_thr & width_check(Start, End, 100, 1000)]
    if ("Present.in.adult.tissues" %in% names(df))
      df <- df[Present.in.adult.tissues == "yes"]
  }

  # ------------------------------------------------
  # ADAPTIVE RETENTION
  # ------------------------------------------------
  n1 <- nrow(df); pct <- 100 * n1 / n0
  if (type == "DHS") {
    target_lo <- 25; target_hi <- 40
    if (pct < target_lo) {
      q_sig <- 0.05
    } else if (pct > target_hi) {
      q_sig <- 0.30
    }
    sig_thr <- safe_quantile(df$mean_signal, q_sig)
    q_score_dhs <- 0.10
    score_thr <- safe_quantile(df$score, q_score_dhs)

    cat(sprintf("DHS thresholds (adaptive): mean_signal > %.4f ; score > %.4f\n",
                sig_thr, score_thr))

    n_a <- nrow(df)
    df1 <- df[mean_signal > sig_thr]
    cat(sprintf("  After mean_signal: %d (%.1f%%)\n", nrow(df1), 100*nrow(df1)/n_a))
    df2 <- df1[score > score_thr]
    cat(sprintf("  After score:       %d (%.1f%%)\n", nrow(df2), 100*nrow(df2)/n_a))
    df3 <- df2[width_check(Start, End, 40, 1500)]
    cat(sprintf("  After width:       %d (%.1f%%)\n", nrow(df3), 100*nrow(df3)/n_a))
    df <- df3
    n1 <- nrow(df); pct <- 100 * n1 / n0
    cat(sprintf("Adaptive DHS quantile %.2f → %.1f%% retained\n", q_sig, pct))

  } else if (type == "ATAC") {
    target_lo <- 40; target_hi <- 60
    if (pct < target_lo) {
      q_score <- 0.35
    } else if (pct > target_hi) {
      q_score <- 0.15
    }
    score_thr <- safe_quantile(df$score, q_score)
    df <- df[score > score_thr & fold.change > 3 & log10qvalue > 1.3]
    df <- df[width_check(Start, End, 100, 1000)]
    n1 <- nrow(df); pct <- 100 * n1 / n0
    cat(sprintf("Adaptive ATAC quantile %.2f → %.1f%% retained\n", q_score, pct))
  }

  # ------------------------------------------------
  # BLACKLIST FILTER (optional fractional)
  # ------------------------------------------------
  if (!is.null(blk)) {
    gr_tmp <- GRanges(seqnames=df$Chrom, ranges=IRanges(start=df$Start, end=df$End))
    ho <- findOverlaps(gr_tmp, blk, ignore.strand = TRUE)
    frac <- rep(0, length(gr_tmp))
    if (length(ho)) {
      inter_w <- width(pintersect(gr_tmp[queryHits(ho)], blk[subjectHits(ho)], drop.nohit.ranges = TRUE))
      frac[unique(queryHits(ho))] <- tapply(inter_w, queryHits(ho), sum) / width(gr_tmp[unique(queryHits(ho))])
    }
    keep <- frac < 0.5
    df <- df[keep]
  }

  # ------------------------------------------------
  # SUMMARIES + SAVE
  # ------------------------------------------------
  cat(summary_stats(df, paste(type, "(filtered)")), "\n")

  out_dir <- if (type == "DHS") qc_dhs else qc_atac
  fwrite(df, file.path(out_dir, paste0(base_name, "_filtered.bed")), sep="\t")

  gr_label <- paste0(base_name, "_", type)
  gr <- to_gr(df, gr_label)
  saveRDS(gr, file.path(out_dir, paste0(base_name, "_filtered.hg19.rds")))

  wstats <- width_stats(df$Start, df$End)
  used_q <- if (type == "DHS") q_sig else q_score
  qc_summary <<- rbind(
    qc_summary,
    data.table(
      dataset = type,
      tissue = base_name,
      initial_peaks = n0,
      retained_peaks = nrow(df),
      retained_pct = round(100 * nrow(df) / n0, 1),
      used_quantile = used_q,
      median_width = wstats$median_width,
      iqr_width = wstats$iqr_width,
      mean_signal_filtered = if ("mean_signal" %in% names(df))
        mean(df$mean_signal, na.rm=TRUE) else NA_real_,
      mean_score_filtered = mean(df$score, na.rm=TRUE)
    ),
    fill = TRUE
  )

  cat(sprintf("✅ %s QC complete: %d / %d peaks retained (%.1f%%)\n",
              type, nrow(df), n0, 100 * nrow(df) / n0))
}

# ------------------------------------------------
# Run loops
# ------------------------------------------------
for (f in dhs_files)  process_cre(f, "DHS")
for (f in atac_files) process_cre(f, "ATAC")

sink()  # close report

# ------------------------------------------------
# Save summary
# ------------------------------------------------
qc_path <- file.path(qc_root, "QC_summary2.csv")
fwrite(qc_summary, qc_path)
cat("\n✅ All QC complete.\n")
cat("QC summary saved to:", qc_path, "\n")
cat("QC report written to:", report_path, "\n")


Warning message:
“package ‘data.table’ was built under R version 4.3.3”
Warning message:
“package ‘GenomicRanges’ was built under R version 4.3.3”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.3.2”
Warning message:
“package ‘S4Vectors’ was built under R version 4.3.3”
Warning message:
“package ‘IRanges’ was built under R version 4.3.3”
Warning message:
“package ‘GenomeInfoDb’ was built under R version 4.3.2”
Warning message:
“package ‘rtracklayer’ was built under R version 4.3.3”


✅ Blacklist loaded: 834 regions
=========== DHS & ATAC QC REPORT ===========


--- DHS QC: Cancer_epithelial ---
[DHS (raw)] mean_signal: 0.618 (IQR 0.207–0.729), score: 0.036 (IQR 0.003–0.040) 
DHS thresholds (initial): mean_signal > 0.1201 ; score > 0.0011
  After mean_signal: 113972 (90.0%)
  After score:       113297 (89.5%)
  After width:       112694 (89.0%)
DHS thresholds (adaptive): mean_signal > 0.2876 ; score > 0.0018
  After mean_signal: 78886 (70.0%)
  After score:       78886 (70.0%)
  After width:       78886 (70.0%)
Adaptive DHS quantile 0.30 → 62.3% retained
[DHS (filtered)] mean_signal: 0.887 (IQR 0.415–1.012), score: 0.054 (IQR 0.012–0.065) 
✅ DHS QC complete: 78475 / 126641 peaks retained (62.0%)

--- DHS QC: Cardiac ---
[DHS (raw)] mean_signal: 0.622 (IQR 0.286–0.754), score: 0.065 (IQR 0.012–0.077) 
DHS thresholds (initial): mean_signal > 0.1956 ; score > 0.0067
  After mean_signal: 91170 (90.0%)
  After score:       88809 (87.6%)
  After width:       88312 (87.1%)

ERROR: Error: Object 'fold.change' not found. Perhaps you intended [fold-change]


In [4]:
#!/usr/bin/env Rscript
# ================================================================
# DHS & ATAC-Seq Quality Control Pipeline (v7 – Adaptive + Metadata)
# ================================================================
suppressPackageStartupMessages({
  library(data.table)
  library(GenomicRanges)
  library(rtracklayer)
})

# ------------------------------------------------
# Utility helpers
# ------------------------------------------------
safe_quantile <- function(x, q, na.rm = TRUE) {
  if (length(x) == 0 || all(is.na(x))) return(NA_real_)
  quantile(x, q, na.rm = na.rm)
}

width_check <- function(start, end, minw, maxw) {
  w <- end - start
  w >= minw & w <= maxw
}

width_stats <- function(start, end) {
  w <- end - start
  list(median_width = median(w), iqr_width = IQR(w))
}

summary_stats <- function(df, label) {
  has_ms <- "mean_signal" %in% names(df) && is.numeric(df$mean_signal)
  ms_mean <- if (has_ms) mean(df$mean_signal, na.rm = TRUE) else NA_real_
  ms_q25  <- if (has_ms) quantile(df$mean_signal, 0.25, na.rm = TRUE) else NA_real_
  ms_q75  <- if (has_ms) quantile(df$mean_signal, 0.75, na.rm = TRUE) else NA_real_
  sc_mean <- mean(df$score, na.rm = TRUE)
  sc_q25  <- quantile(df$score, 0.25, na.rm = TRUE)
  sc_q75  <- quantile(df$score, 0.75, na.rm = TRUE)
  sprintf("[%s] mean_signal: %.3f (IQR %.3f–%.3f), score: %.3f (IQR %.3f–%.3f)",
          label, ms_mean, ms_q25, ms_q75, sc_mean, sc_q25, sc_q75)
}

# ---- preserve metadata ------------------------------------------
to_gr <- function(df, name_label) {
  gr <- GRanges(seqnames = df$Chrom,
                ranges   = IRanges(start=df$Start, end=df$End),
                strand   = "*")
  meta_cols <- setdiff(names(df), c("Chrom", "Start", "End"))
  mcols(gr) <- as.data.frame(df[, meta_cols, with = FALSE])
  mcols(gr)$CRE_source <- name_label
  gr
}

# ------------------------------------------------
# Input directories
# ------------------------------------------------
dhs_dir  <- "../ref/CRE_sites/DHS_sites2"
atac_dir <- "../ref/CRE_sites/scATAC_sites2"
dhs_files  <- list.files(dhs_dir,  pattern="\\.bed$", full.names=TRUE)
atac_files <- list.files(atac_dir, pattern="\\.bed$", full.names=TRUE)

# ------------------------------------------------
# Output directories
# ------------------------------------------------
qc_root <- "../ref/CRE_sites_QC"
qc_dhs  <- file.path(qc_root, "DHS_sites_QC2")
qc_atac <- file.path(qc_root, "scATAC_sites_QC2")
dir.create(qc_root, recursive = TRUE, showWarnings = FALSE)
dir.create(qc_dhs,  recursive = TRUE, showWarnings = FALSE)
dir.create(qc_atac, recursive = TRUE, showWarnings = FALSE)

# ------------------------------------------------
# Load blacklist (optional)
# ------------------------------------------------
blk_path <- "../ref/hg19-blacklist.v2.bed"
blk <- if (file.exists(blk_path)) import(blk_path) else NULL
if (!is.null(blk)) cat("✅ Blacklist loaded:", length(blk), "regions\n")

# ------------------------------------------------
# Initialize outputs
# ------------------------------------------------
qc_summary <- data.table(
  dataset = character(),
  tissue = character(),
  initial_peaks = integer(),
  retained_peaks = integer(),
  retained_pct = numeric(),
  used_quantile = numeric(),
  median_width = numeric(),
  iqr_width = numeric(),
  mean_signal_filtered = numeric(),
  mean_score_filtered = numeric()
)

report_path <- file.path(qc_root, "QC_report2.txt")
sink(report_path, append = FALSE)
cat("=========== DHS & ATAC QC REPORT ===========\n\n")

# ------------------------------------------------
# Adaptive QC function
# ------------------------------------------------
process_cre <- function(path, type) {
  df <- fread(path, sep="\t", header=TRUE)
  base_name <- tools::file_path_sans_ext(basename(path))
  cat(sprintf("\n--- %s QC: %s ---\n", type, base_name))

  n0 <- nrow(df)
  df <- df[!is.na(Start) & !is.na(End) & Start < End]
  df <- unique(df)
  cat(summary_stats(df, paste(type, "(raw)")), "\n")

  # ------------------------------------------------
  # INITIAL FILTERS
  # ------------------------------------------------
  if (type == "DHS") {
    q_sig <- 0.10
    sig_thr <- safe_quantile(df$mean_signal, q_sig)

    q_score_dhs <- 0.10
    score_thr   <- safe_quantile(df$score, q_score_dhs)

    cat(sprintf("DHS thresholds (initial): mean_signal > %.4f ; score > %.4f\n",
                sig_thr, score_thr))

    n_a <- nrow(df)
    df1 <- df[mean_signal > sig_thr]
    cat(sprintf("  After mean_signal: %d (%.1f%%)\n", nrow(df1), 100*nrow(df1)/n_a))
    df2 <- df1[score > score_thr]
    cat(sprintf("  After score:       %d (%.1f%%)\n", nrow(df2), 100*nrow(df2)/n_a))
    df3 <- df2[width_check(Start, End, 40, 1500)]
    cat(sprintf("  After width:       %d (%.1f%%)\n", nrow(df3), 100*nrow(df3)/n_a))
    df <- df3

  } else if (type == "ATAC") {
    q_score <- 0.25
    df <- df[`fold-change` > 3 & log10qvalue > 1.3]
    score_thr <- safe_quantile(df$score, q_score)
    df <- df[score > score_thr & width_check(Start, End, 100, 1000)]
    if ("Present.in.adult.tissues" %in% names(df))
      df <- df[Present.in.adult.tissues == "yes"]
  }

  # ------------------------------------------------
  # ADAPTIVE RETENTION
  # ------------------------------------------------
  n1 <- nrow(df); pct <- 100 * n1 / n0
  if (type == "DHS") {
    target_lo <- 25; target_hi <- 40
    if (pct < target_lo) {
      q_sig <- 0.05
    } else if (pct > target_hi) {
      q_sig <- 0.30
    }
    sig_thr <- safe_quantile(df$mean_signal, q_sig)
    q_score_dhs <- 0.10
    score_thr <- safe_quantile(df$score, q_score_dhs)

    cat(sprintf("DHS thresholds (adaptive): mean_signal > %.4f ; score > %.4f\n",
                sig_thr, score_thr))

    n_a <- nrow(df)
    df1 <- df[mean_signal > sig_thr]
    cat(sprintf("  After mean_signal: %d (%.1f%%)\n", nrow(df1), 100*nrow(df1)/n_a))
    df2 <- df1[score > score_thr]
    cat(sprintf("  After score:       %d (%.1f%%)\n", nrow(df2), 100*nrow(df2)/n_a))
    df3 <- df2[width_check(Start, End, 40, 1500)]
    cat(sprintf("  After width:       %d (%.1f%%)\n", nrow(df3), 100*nrow(df3)/n_a))
    df <- df3
    n1 <- nrow(df); pct <- 100 * n1 / n0
    cat(sprintf("Adaptive DHS quantile %.2f → %.1f%% retained\n", q_sig, pct))

  } else if (type == "ATAC") {
    target_lo <- 40; target_hi <- 60
    if (pct < target_lo) {
      q_score <- 0.35
    } else if (pct > target_hi) {
      q_score <- 0.15
    }
    score_thr <- safe_quantile(df$score, q_score)
    df <- df[score > score_thr & `fold-change` > 3 & log10qvalue > 1.3]
    df <- df[width_check(Start, End, 100, 1000)]
    n1 <- nrow(df); pct <- 100 * n1 / n0
    cat(sprintf("Adaptive ATAC quantile %.2f → %.1f%% retained\n", q_score, pct))
  }

  # ------------------------------------------------
  # BLACKLIST FILTER (optional fractional)
  # ------------------------------------------------
  if (!is.null(blk)) {
    gr_tmp <- GRanges(seqnames=df$Chrom, ranges=IRanges(start=df$Start, end=df$End))
    ho <- findOverlaps(gr_tmp, blk, ignore.strand = TRUE)
    frac <- rep(0, length(gr_tmp))
    if (length(ho)) {
      inter_w <- width(pintersect(gr_tmp[queryHits(ho)], blk[subjectHits(ho)], drop.nohit.ranges = TRUE))
      frac[unique(queryHits(ho))] <- tapply(inter_w, queryHits(ho), sum) / width(gr_tmp[unique(queryHits(ho))])
    }
    keep <- frac < 0.5
    df <- df[keep]
  }

  # ------------------------------------------------
  # SUMMARIES + SAVE
  # ------------------------------------------------
  cat(summary_stats(df, paste(type, "(filtered)")), "\n")

  out_dir <- if (type == "DHS") qc_dhs else qc_atac
  fwrite(df, file.path(out_dir, paste0(base_name, "_filtered.bed")), sep="\t")

  gr_label <- paste0(base_name, "_", type)
  gr <- to_gr(df, gr_label)
  saveRDS(gr, file.path(out_dir, paste0(base_name, "_filtered.hg19.rds")))

  wstats <- width_stats(df$Start, df$End)
  used_q <- if (type == "DHS") q_sig else q_score
  qc_summary <<- rbind(
    qc_summary,
    data.table(
      dataset = type,
      tissue = base_name,
      initial_peaks = n0,
      retained_peaks = nrow(df),
      retained_pct = round(100 * nrow(df) / n0, 1),
      used_quantile = used_q,
      median_width = wstats$median_width,
      iqr_width = wstats$iqr_width,
      mean_signal_filtered = if ("mean_signal" %in% names(df))
        mean(df$mean_signal, na.rm=TRUE) else NA_real_,
      mean_score_filtered = mean(df$score, na.rm=TRUE)
    ),
    fill = TRUE
  )

  cat(sprintf("✅ %s QC complete: %d / %d peaks retained (%.1f%%)\n",
              type, nrow(df), n0, 100 * nrow(df) / n0))
}

# ------------------------------------------------
# Run loops
# ------------------------------------------------
for (f in dhs_files)  process_cre(f, "DHS")
for (f in atac_files) process_cre(f, "ATAC")

sink()  # close report

# ------------------------------------------------
# Save summary
# ------------------------------------------------
qc_path <- file.path(qc_root, "QC_summary2.csv")
fwrite(qc_summary, qc_path)
cat("\n✅ All QC complete.\n")
cat("QC summary saved to:", qc_path, "\n")
cat("QC report written to:", report_path, "\n")

✅ Blacklist loaded: 834 regions
=========== DHS & ATAC QC REPORT ===========


--- DHS QC: Cancer_epithelial ---
[DHS (raw)] mean_signal: 0.618 (IQR 0.207–0.729), score: 0.036 (IQR 0.003–0.040) 
DHS thresholds (initial): mean_signal > 0.1201 ; score > 0.0011
  After mean_signal: 113972 (90.0%)
  After score:       113297 (89.5%)
  After width:       112694 (89.0%)
DHS thresholds (adaptive): mean_signal > 0.2876 ; score > 0.0018
  After mean_signal: 78886 (70.0%)
  After score:       78886 (70.0%)
  After width:       78886 (70.0%)
Adaptive DHS quantile 0.30 → 62.3% retained
[DHS (filtered)] mean_signal: 0.887 (IQR 0.415–1.012), score: 0.054 (IQR 0.012–0.065) 
✅ DHS QC complete: 78475 / 126641 peaks retained (62.0%)

--- DHS QC: Cardiac ---
[DHS (raw)] mean_signal: 0.622 (IQR 0.286–0.754), score: 0.065 (IQR 0.012–0.077) 
DHS thresholds (initial): mean_signal > 0.1956 ; score > 0.0067
  After mean_signal: 91170 (90.0%)
  After score:       88809 (87.6%)
  After width:       88312 (87.1%)

## Step 2: Quality control

In [1]:
#!/usr/bin/env Rscript
# ================================================================
# DHS, ATAC, and TCGA ATAC QC Pipeline (v8 – Adaptive + Metadata + MinPeak)
# ================================================================

suppressPackageStartupMessages({
  library(data.table)
  library(GenomicRanges)
  library(rtracklayer)
})

# ------------------------------------------------
# Utility functions
# ------------------------------------------------
safe_quantile <- function(x, q, na.rm = TRUE) {
  if (length(x) == 0 || all(is.na(x))) return(NA_real_)
  quantile(x, q, na.rm = na.rm)
}

width_check <- function(start, end, minw, maxw) {
  w <- end - start
  w >= minw & w <= maxw
}

width_stats <- function(start, end) {
  w <- end - start
  list(median_width = median(w), iqr_width = IQR(w))
}

summary_stats <- function(df, label) {
  has_ms <- "mean_signal" %in% names(df) && is.numeric(df$mean_signal)
  ms_mean <- if (has_ms) mean(df$mean_signal, na.rm = TRUE) else NA_real_
  ms_q25  <- if (has_ms) quantile(df$mean_signal, 0.25, na.rm = TRUE) else NA_real_
  ms_q75  <- if (has_ms) quantile(df$mean_signal, 0.75, na.rm = TRUE) else NA_real_
  sc_mean <- mean(df$score, na.rm = TRUE)
  sc_q25  <- quantile(df$score, 0.25, na.rm = TRUE)
  sc_q75  <- quantile(df$score, 0.75, na.rm = TRUE)
  sprintf("[%s] mean_signal: %.3f (IQR %.3f–%.3f), score: %.3f (IQR %.3f–%.3f)",
          label, ms_mean, ms_q25, ms_q75, sc_mean, sc_q25, sc_q75)
}

to_gr <- function(df, name_label) {
  gr <- GRanges(seqnames = df$Chrom,
                ranges   = IRanges(start=df$Start, end=df$End),
                strand   = "*")
  meta_cols <- setdiff(names(df), c("Chrom", "Start", "End"))
  mcols(gr) <- as.data.frame(df[, meta_cols, with = FALSE])
  mcols(gr)$CRE_source <- name_label
  gr
}

# ------------------------------------------------
# Input directories
# ------------------------------------------------
dhs_dir   <- "../ref/CRE_sites/DHS_sites2"
atac_dir  <- "../ref/CRE_sites/scATAC_sites2"
tcga_dir  <- "../ref/CRE_sites/TCGA_sites2"

dhs_files  <- list.files(dhs_dir,  pattern="\\.bed$", full.names=TRUE)
atac_files <- list.files(atac_dir, pattern="\\.bed$", full.names=TRUE)
tcga_files <- list.files(tcga_dir, pattern="\\.bed$", full.names=TRUE)

# ------------------------------------------------
# Output directories
# ------------------------------------------------
qc_root <- "../ref/CRE_sites_QC"
qc_dhs  <- file.path(qc_root, "DHS_sites_QC2")
qc_atac <- file.path(qc_root, "scATAC_sites_QC2")
qc_tcga <- file.path(qc_root, "TCGA_sites_QC2")
dir.create(qc_root, recursive = TRUE, showWarnings = FALSE)
dir.create(qc_dhs,  recursive = TRUE, showWarnings = FALSE)
dir.create(qc_atac, recursive = TRUE, showWarnings = FALSE)
dir.create(qc_tcga, recursive = TRUE, showWarnings = FALSE)

# ------------------------------------------------
# Optional blacklist
# ------------------------------------------------
blk_path <- "../ref/hg19-blacklist.v2.bed"
blk <- if (file.exists(blk_path)) import(blk_path) else NULL
if (!is.null(blk)) cat("✅ Blacklist loaded:", length(blk), "regions\n")

# ------------------------------------------------
# Initialize outputs
# ------------------------------------------------
qc_summary <- data.table(
  dataset = character(),
  tissue = character(),
  initial_peaks = integer(),
  retained_peaks = integer(),
  retained_pct = numeric(),
  used_quantile = numeric(),
  median_width = numeric(),
  iqr_width = numeric(),
  mean_signal_filtered = numeric(),
  mean_score_filtered = numeric()
)

report_path <- file.path(qc_root, "QC_report_TCGA_v8.txt")
sink(report_path, append = FALSE)
cat("=========== DHS, ATAC, TCGA QC REPORT (v8) ===========\n\n")

# ------------------------------------------------
# Adaptive QC core function
# ------------------------------------------------
process_cre <- function(path, type) {
  df <- fread(path, sep="\t", header=TRUE)
  base_name <- tools::file_path_sans_ext(basename(path))
  cat(sprintf("\n--- %s QC: %s ---\n", type, base_name))

  n0 <- nrow(df)
  df <- df[!is.na(Start) & !is.na(End) & Start < End]
  df <- unique(df)
  cat(summary_stats(df, paste(type, "(raw)")), "\n")

  # -------------------------------
  # INITIAL FILTERS
  # -------------------------------
  if (type == "DHS") {
    q_sig <- 0.10
    sig_thr <- safe_quantile(df$mean_signal, q_sig)
    q_score <- 0.10
    score_thr <- safe_quantile(df$score, q_score)
    df <- df[mean_signal > sig_thr & score > score_thr]
    df <- df[width_check(Start, End, 40, 1500)]

  } else if (type == "ATAC") {
    q_score <- 0.25
    df <- df[`fold-change` > 3 & log10qvalue > 1.3]
    score_thr <- safe_quantile(df$score, q_score)
    df <- df[score > score_thr & width_check(Start, End, 100, 1000)]

  } else if (type == "TCGA") {
    q_score <- 0.25
    df <- df[width_check(Start, End, 100, 1000)]
    score_thr <- safe_quantile(df$score, q_score)
    df <- df[score > score_thr]
    if ("percentGC" %in% names(df))
      df <- df[percentGC > 0.3 & percentGC < 0.7]
  }

  # -------------------------------
  # ADAPTIVE RETENTION
  # -------------------------------
  n1 <- nrow(df); pct <- 100 * n1 / n0
  target_lo <- if (type == "DHS") 25 else 40
  target_hi <- if (type == "DHS") 40 else 60
  if (pct < target_lo) {
    q_score <- 0.35
  } else if (pct > target_hi) {
    q_score <- 0.15
  }
  score_thr <- safe_quantile(df$score, q_score)
  df <- df[score > score_thr]
  df <- df[width_check(Start, End, 100, 1000)]
  n1 <- nrow(df); pct <- 100 * n1 / n0
  cat(sprintf("%s adaptive quantile %.2f → %.1f%% retained\n", type, q_score, pct))

  # -------------------------------
  # BLACKLIST FILTER
  # -------------------------------
  if (!is.null(blk)) {
    gr_tmp <- GRanges(seqnames=df$Chrom, ranges=IRanges(start=df$Start, end=df$End))
    ho <- findOverlaps(gr_tmp, blk, ignore.strand = TRUE)
    frac <- rep(0, length(gr_tmp))
    if (length(ho)) {
      inter_w <- width(pintersect(gr_tmp[queryHits(ho)], blk[subjectHits(ho)], drop.nohit.ranges = TRUE))
      frac[unique(queryHits(ho))] <- tapply(inter_w, queryHits(ho), sum) / width(gr_tmp[unique(queryHits(ho))])
    }
    df <- df[frac < 0.5]
  }

  # -------------------------------
  # ENFORCE MINIMUM RETAINED PEAKS
  # -------------------------------
  if (nrow(df) < 12000) {
    cat(sprintf("⚠️  %s has only %d peaks after filtering — retaining top 12,000 by score.\n",
                base_name, nrow(df)))
    if ("mean_signal" %in% names(df)) {
      setorder(df, -mean_signal)
    } else {
      setorder(df, -score)
    }
    if (nrow(df) > 0) df <- df[1:min(12000, nrow(df)), ]
  }

  # -------------------------------
  # SUMMARIES + SAVE
  # -------------------------------
  cat(summary_stats(df, paste(type, "(filtered)")), "\n")

  out_dir <- switch(type,
                    "DHS" = qc_dhs,
                    "ATAC" = qc_atac,
                    "TCGA" = qc_tcga)

  fwrite(df, file.path(out_dir, paste0(base_name, "_filtered.bed")), sep="\t")

  gr_label <- paste0(base_name, "_", type)
  gr <- to_gr(df, gr_label)
  saveRDS(gr, file.path(out_dir, paste0(base_name, "_filtered.hg19.rds")))

  wstats <- width_stats(df$Start, df$End)
  qc_summary <<- rbind(
    qc_summary,
    data.table(
      dataset = type,
      tissue = base_name,
      initial_peaks = n0,
      retained_peaks = nrow(df),
      retained_pct = round(100 * nrow(df) / n0, 1),
      used_quantile = q_score,
      median_width = wstats$median_width,
      iqr_width = wstats$iqr_width,
      mean_signal_filtered = if ("mean_signal" %in% names(df))
        mean(df$mean_signal, na.rm=TRUE) else NA_real_,
      mean_score_filtered = mean(df$score, na.rm=TRUE)
    ),
    fill = TRUE
  )

  cat(sprintf("✅ %s QC complete: %d / %d peaks retained (%.1f%%)\n",
              type, nrow(df), n0, 100 * nrow(df) / n0))
}

# ------------------------------------------------
# Run QC loops
# ------------------------------------------------
for (f in dhs_files)  process_cre(f, "DHS")
for (f in atac_files) process_cre(f, "ATAC")
for (f in tcga_files) process_cre(f, "TCGA")

sink()

# ------------------------------------------------
# Save summary
# ------------------------------------------------
qc_path <- file.path(qc_root, "QC_summary_TCGA_v8.csv")
fwrite(qc_summary, qc_path)
cat("\n✅ All QC complete.\n")
cat("QC summary saved to:", qc_path, "\n")
cat("QC report written to:", report_path, "\n")


Warning message:
“package ‘data.table’ was built under R version 4.3.3”
Warning message:
“package ‘GenomicRanges’ was built under R version 4.3.3”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.3.2”
Warning message:
“package ‘S4Vectors’ was built under R version 4.3.3”
Warning message:
“package ‘IRanges’ was built under R version 4.3.3”
Warning message:
“package ‘GenomeInfoDb’ was built under R version 4.3.2”
Warning message:
“package ‘rtracklayer’ was built under R version 4.3.3”


✅ Blacklist loaded: 834 regions
=========== DHS, ATAC, TCGA QC REPORT (v8) ===========


--- DHS QC: Cancer_epithelial ---
[DHS (raw)] mean_signal: 0.618 (IQR 0.207–0.729), score: 0.036 (IQR 0.003–0.040) 
DHS adaptive quantile 0.15 → 72.0% retained
[DHS (filtered)] mean_signal: 0.783 (IQR 0.332–0.907), score: 0.049 (IQR 0.011–0.057) 
✅ DHS QC complete: 90662 / 126641 peaks retained (71.6%)

--- DHS QC: Cardiac ---
[DHS (raw)] mean_signal: 0.622 (IQR 0.286–0.754), score: 0.065 (IQR 0.012–0.077) 
DHS adaptive quantile 0.15 → 70.7% retained
[DHS (filtered)] mean_signal: 0.771 (IQR 0.407–0.928), score: 0.088 (IQR 0.024–0.113) 
✅ DHS QC complete: 71363 / 101340 peaks retained (70.4%)

--- DHS QC: Digestive ---
[DHS (raw)] mean_signal: 0.681 (IQR 0.279–0.855), score: 0.042 (IQR 0.008–0.043) 
DHS adaptive quantile 0.15 → 70.2% retained
[DHS (filtered)] mean_signal: 0.859 (IQR 0.422–1.062), score: 0.058 (IQR 0.015–0.063) 
✅ DHS QC complete: 79721 / 114144 peaks retained (69.8%)

--- DHS QC: Ly

In [2]:
#!/usr/bin/env Rscript
# ================================================================
# CRE QC Validation — Check metadata integrity for DHS, ATAC, and TCGA
# ================================================================

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(data.table)
})

# ------------------------------------------------
# Load one filtered DHS, ATAC, and TCGA
# ------------------------------------------------
dhs_example  <- readRDS("../ref/CRE_sites_QC/DHS_sites_QC2/Cardiac_filtered.hg19.rds")
atac_example <- readRDS("../ref/CRE_sites_QC/scATAC_sites_QC2/Adrenal_Cortical_filtered.hg19.rds")
tcga_example <- readRDS("../ref/CRE_sites_QC/TCGA_sites_QC2/LUAD_ATAC_Peaks_hg19_filtered.hg19.rds")

# ------------------------------------------------
# Inspect structure and metadata columns
# ------------------------------------------------
cat("✅ DHS object structure:\n")
dhs_example
cat("\nMetadata columns (DHS):\n")
print(head(mcols(dhs_example)))

cat("\n\n✅ ATAC object structure:\n")
atac_example
cat("\nMetadata columns (ATAC):\n")
print(head(mcols(atac_example)))

cat("\n\n✅ TCGA object structure:\n")
tcga_example
cat("\nMetadata columns (TCGA):\n")
print(head(mcols(tcga_example)))

# ------------------------------------------------
# Simple QC validation checks
# ------------------------------------------------

# Width summaries
dhs_widths  <- width(dhs_example)
atac_widths <- width(atac_example)
tcga_widths <- width(tcga_example)

cat("\n--- WIDTH SUMMARY ---\n")
cat(sprintf("DHS  median width: %.1f bp (IQR %.1f–%.1f)\n",
            median(dhs_widths),
            quantile(dhs_widths, 0.25),
            quantile(dhs_widths, 0.75)))
cat(sprintf("ATAC median width: %.1f bp (IQR %.1f–%.1f)\n",
            median(atac_widths),
            quantile(atac_widths, 0.25),
            quantile(atac_widths, 0.75)))
cat(sprintf("TCGA median width: %.1f bp (IQR %.1f–%.1f)\n",
            median(tcga_widths),
            quantile(tcga_widths, 0.25),
            quantile(tcga_widths, 0.75)))

# ------------------------------------------------
# Signal/score distributions
# ------------------------------------------------
if ("mean_signal" %in% colnames(mcols(dhs_example))) {
  cat("\n--- DHS mean_signal distribution ---\n")
  print(summary(mcols(dhs_example)$mean_signal))
}

cat("\n--- DHS score distribution ---\n")
print(summary(mcols(dhs_example)$score))

cat("\n--- ATAC score distribution ---\n")
print(summary(mcols(atac_example)$score))

cat("\n--- TCGA score distribution ---\n")
print(summary(mcols(tcga_example)$score))

if ("percentGC" %in% colnames(mcols(tcga_example))) {
  cat("\n--- TCGA percentGC distribution ---\n")
  print(summary(mcols(tcga_example)$percentGC))
}

# ------------------------------------------------
# Peak counts
# ------------------------------------------------
cat(sprintf("\nDHS  peaks retained: %d\n", length(dhs_example)))
cat(sprintf("ATAC peaks retained: %d\n", length(atac_example)))
cat(sprintf("TCGA peaks retained: %d\n", length(tcga_example)))

# ------------------------------------------------
# Metadata consistency checks
# ------------------------------------------------
cat("\n--- METADATA CONSISTENCY ---\n")

check_metadata <- function(gr, label) {
  cols <- colnames(mcols(gr))
  cat(sprintf("\n%s metadata columns (%d):\n", label, length(cols)))
  print(cols)
}

check_metadata(dhs_example, "DHS")
check_metadata(atac_example, "ATAC")
check_metadata(tcga_example, "TCGA")

cat("\n✅ Metadata inspection complete — all datasets loaded successfully.\n")


✅ DHS object structure:


GRanges object with 71363 ranges and 10 metadata columns:
          seqnames            ranges strand | summit_hg19 core_start_hg19
             <Rle>         <IRanges>  <Rle> |   <integer>       <integer>
      [1]     chr1     766720-766940      * |      766830          766790
      [2]     chr1     767517-767786      * |      767660          767569
      [3]     chr1     800112-800340      * |      800230          800176
      [4]     chr1     800764-800975      * |      800860          800850
      [5]     chr1     801080-801357      * |      801230          801190
      ...      ...               ...    ... .         ...             ...
  [71359]     chrY 22584726-22584926      * |    22584836        22584776
  [71360]     chrY 22646646-22646806      * |    22646716        22646696
  [71361]     chrY 22743688-22743866      * |    22743796        22743796
  [71362]     chrY 23762106-23762326      * |    23762226        23762198
  [71363]     chrY 28466307-28466450      * |    28466


Metadata columns (DHS):
DataFrame with 6 rows and 10 columns
  summit_hg19 core_start_hg19 core_end_hg19        name mean_signal peak.count
    <integer>       <integer>     <integer> <character>   <numeric>  <integer>
1      766830          766790        766857    1.103021    1.550273         27
2      767660          767569        767751    1.103026    0.406669         20
3      800230          800176        800230    1.103138    0.396565          7
4      800860          800850        800942   1.1031405    0.386003          6
5      801230          801190        801310   1.1031415    1.518455         34
6      801370          801370        801370   1.1031424    0.603776          1
    component  position     score  CRE_source
  <character> <integer> <numeric> <character>
1     Cardiac    831450 0.1376938 Cardiac_DHS
2     Cardiac    832280 0.0766045 Cardiac_DHS
3     Cardiac    864850 0.0326481 Cardiac_DHS
4     Cardiac    865480 0.0289452 Cardiac_DHS
5     Cardiac    865850 0.1601

GRanges object with 21000 ranges and 21 metadata columns:
          seqnames              ranges strand | ATAC_summit_hg19
             <Rle>           <IRanges>  <Rle> |        <integer>
      [1]     chr4       830468-830868      * |           830668
      [2]     chrX 149369167-149369567      * |        149369367
      [3]     chr2   85672840-85673240      * |         85673040
      [4]     chrX 118175141-118175541      * |        118175341
      [5]     chr1   26363015-26363415      * |         26363215
      ...      ...                 ...    ... .              ...
  [20996]     chr2   42866046-42866446      * |         42866246
  [20997]     chr7 155592196-155592596      * |        155592396
  [20998]    chr12 120868765-120869165      * |        120868965
  [20999]     chr1   75992869-75993269      * |         75993069
  [21000]     chr1 156238575-156238975      * |        156238775
                     name     score      strand fold-change log10pvalue
              <character>


Metadata columns (ATAC):
DataFrame with 6 rows and 21 columns
  ATAC_summit_hg19            name     score      strand fold-change
         <integer>     <character> <integer> <character>   <numeric>
1           830668  NA_peak_133245     62838           .     14.5359
2        149369367 NA_peak_200127b     44166           .     16.1834
3         85673040   NA_peak_48891     34571           .     19.0606
4        118175341 NA_peak_199039b     34401           .     16.8827
5         26363215    NA_peak_1690     34292           .     12.9491
6         24527114   NA_peak_46624     33537           .     33.3958
  log10pvalue log10qvalue summit_distance                     ID  position
    <numeric>   <numeric>       <integer>            <character> <integer>
1     6287.69     6283.86             200     chr4_836680_837080    836880
2     4420.11     4416.65             200 chrX_150200937_15020.. 150201137
3     3461.10     3457.16             200 chr2_85445717_85446117  85445917
4     3443

GRanges object with 81664 ranges and 6 metadata columns:
          seqnames            ranges strand |        name     score  annotation
             <Rle>         <IRanges>  <Rle> | <character> <numeric> <character>
      [1]     chr1     949616-950117      * |     LUAD_39   4.19390      3' UTR
      [2]     chr1   1227132-1227633      * |    LUAD_114   5.67114      3' UTR
      [3]     chr1   1376181-1376682      * |    LUAD_166   8.74222      3' UTR
      [4]     chr1   1376915-1377416      * |    LUAD_167   6.83556      3' UTR
      [5]     chr1   2120892-2121393      * |    LUAD_266  30.03738      3' UTR
      ...      ...               ...    ... .         ...       ...         ...
  [81660]     chrY 15017361-15017862      * | LUAD_139190   5.21296    Promoter
  [81661]     chrY 15591712-15592213      * | LUAD_139196  10.79964    Promoter
  [81662]     chrY 21237760-21238261      * | LUAD_139210   5.52901    Promoter
  [81663]     chrY 21906391-21906892      * | LUAD_139220   5.2


Metadata columns (TCGA):
DataFrame with 6 rows and 6 columns
         name     score  annotation percentGC percentAT             CRE_source
  <character> <numeric> <character> <numeric> <numeric>            <character>
1     LUAD_39   4.19390      3' UTR  0.624750  0.375250 LUAD_ATAC_Peaks_hg19..
2    LUAD_114   5.67114      3' UTR  0.638723  0.361277 LUAD_ATAC_Peaks_hg19..
3    LUAD_166   8.74222      3' UTR  0.654691  0.345309 LUAD_ATAC_Peaks_hg19..
4    LUAD_167   6.83556      3' UTR  0.648703  0.351297 LUAD_ATAC_Peaks_hg19..
5    LUAD_266  30.03738      3' UTR  0.658683  0.341317 LUAD_ATAC_Peaks_hg19..
6    LUAD_268   5.64774      3' UTR  0.570858  0.429142 LUAD_ATAC_Peaks_hg19..

--- WIDTH SUMMARY ---
DHS  median width: 211.0 bp (IQR 181.0–258.0)
ATAC median width: 401.0 bp (IQR 401.0–401.0)
TCGA median width: 502.0 bp (IQR 502.0–502.0)

--- DHS mean_signal distribution ---
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1956  0.4070  0.5799  0.7707  0.9275 30.4314 

--- DHS 